In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical
import numpy as np

class HyperparameterController(nn.Module):
    def __init__(self):
        super(HyperparameterController, self).__init__()

        # Shared representation layer
        self.shared = nn.Linear(1, 16)

        # Output heads for each hyperparameter logits
        self.act_head = nn.Linear(16, 4)        # 4 activations
        self.lr_head = nn.Linear(16, 5)         # 5 learning rates
        self.layers_head = nn.Linear(16, 5)     # 1 to 5 hidden layers
        self.size_head = nn.Linear(16, 5)       # 5 layer size options

        # Mapping dictionaries to decode network outputs into actual values
        self.act_map = {0: 'relu', 1: 'sigmoid', 2: 'softmax', 3: 'gauss'}
        self.lr_map = {0: 0.01, 1: 0.1, 2: 0.5, 3: 1.0, 4: 2.0}
        self.layers_map = {0: 1, 1: 2, 2: 3, 3: 4, 4: 5}
        self.size_map = {0: 4, 1: 8, 2: 12, 3: 16, 4: 32}

    def forward(self, dummy_input):
        x = torch.relu(self.shared(dummy_input))

        # Get logits for each hyperparameter
        act_logits = self.act_head(x)
        lr_logits = self.lr_head(x)
        layers_logits = self.layers_head(x)
        size_logits = self.size_head(x)

        return act_logits, lr_logits, layers_logits, size_logits

    def sample_hyperparameters(self):
        # Dummy input to trigger the forward pass
        dummy_input = torch.ones(1)
        act_logits, lr_logits, layers_logits, size_logits = self.forward(dummy_input)

        # Create categorical distributions
        act_dist = Categorical(logits=act_logits)
        lr_dist = Categorical(logits=lr_logits)
        layers_dist = Categorical(logits=layers_logits)
        size_dist = Categorical(logits=size_logits)

        # Sample an index for each hyperparameter
        act_idx = act_dist.sample()
        lr_idx = lr_dist.sample()
        layers_idx = layers_dist.sample()
        size_idx = size_dist.sample()

        # Sum the log probabilities for the REINFORCE loss calculation
        log_prob = (act_dist.log_prob(act_idx) +
                    lr_dist.log_prob(lr_idx) +
                    layers_dist.log_prob(layers_idx) +
                    size_dist.log_prob(size_idx))

        # Decode indices into actual hyperparameter values
        params = {
            'activation': self.act_map[act_idx.item()],
            'learning_rate': self.lr_map[lr_idx.item()],
            'num_layers': self.layers_map[layers_idx.item()],
            'layer_size': self.size_map[size_idx.item()]
        }

        return params, log_prob

In [ ]:
def train_child_network(params, x_train, y_train, K, N_epochs=1000):
    mid_size_array = [params['layer_size']] * params['num_layers']

    model, x_train_tensor, y_onehot, optimizer, cost_func = setup_model_and_data(
        x_train, y_train, K,
        mid_size_array,
        params['learning_rate'],
        N_epochs,
        params['activation']
    )

    cost_history = []
    acc_history = []

    # Training loop tracking the trajectory
    for epoch in range(N_epochs):
        optimizer.zero_grad()
        NN_output = model(x_train_tensor)
        cost = cost_func(NN_output, y_onehot)
        cost.backward()
        optimizer.step()

        # Track cost
        cost_history.append(cost.item())

        # Track accuracy (calculating this every epoch adds slight overhead,
        # but is necessary for a trajectory-based reward)
        predicted_class = np.argmax(NN_output.detach().numpy(), axis=1)
        accuracy = np.mean(predicted_class == y_train)
        acc_history.append(accuracy)

    # --- COMPUTE COMPOSITE REWARD ---

    # Base Achievement
    final_acc = acc_history[-1]

    # Convergence Speed
    # High mean accuracy means it reached high performance early
    acc_speed_metric = np.mean(acc_history)
    cost_speed_metric = np.mean(cost_history)

    # Smoothness Penalty
    # Calculate the first discrete difference of the cost curve
    acc_diffs = np.diff(acc_history)
    cost_diffs = np.diff(cost_history)
    # High standard deviation means highly volatile step-to-step changes
    acc_smoothness_penalty = np.std(acc_diffs)
    cost_smoothness_penalty = np.std(cost_diffs)

    #

    # Combine into a final reward scalar
    reward = final_acc + (0.5 * acc_speed_metric - 0.1*cost_speed_metric) - ( 0.1*acc_smoothness_penalty + 0.1*cost_smoothness_penalty)

    return reward

def optimize_hyperparameters_with_nn(x_train, y_train, K, search_episodes=50, N_epochs=1000):
    """
    Uses Policy Gradient (REINFORCE) to train the Controller NN to find optimal hyperparameters.
    """
    controller = HyperparameterController()
    controller_optimizer = torch.optim.Adam(controller.parameters(), lr=0.01)

    baseline_reward = 0.0 # Exponential moving average for reward normalization

    print("Starting Neural Architecture Search...")

    for episode in range(search_episodes):
        controller_optimizer.zero_grad()

        # Controller predicts hyperparameters
        params, log_prob = controller.sample_hyperparameters()

        # Train the child network and get accuracy (Reward)
        reward = train_child_network(params, x_train, y_train, K, N_epochs)

        # Calculate Advantage (Reward - Baseline) to stabilize training
        if episode == 0:
            baseline_reward = reward
        else:
            baseline_reward = 0.9 * baseline_reward + 0.1 * reward

        advantage = reward - baseline_reward

        # Compute Policy Gradient Loss (-log_prob * advantage)
        # We negate it because PyTorch optimizers minimize loss, but we want to maximize reward
        loss = -log_prob * advantage

        # Backpropagate through the Controller
        loss.backward()
        controller_optimizer.step()

        print(f"Episode {episode+1:02d} | Reward: {reward:.4f} | Params: {params}")

    # After search, sample the most confident prediction
    print("\nSearch Complete. Evaluating optimal configuration...")
    with torch.no_grad():
        # Force greedy selection (argmax) instead of sampling for the final output
        dummy_input = torch.ones(1)
        act_logits, lr_logits, layers_logits, size_logits = controller(dummy_input)

        best_params = {
            'activation': controller.act_map[torch.argmax(act_logits).item()],
            'learning_rate': controller.lr_map[torch.argmax(lr_logits).item()],
            'num_layers': controller.layers_map[torch.argmax(layers_logits).item()],
            'layer_size': controller.size_map[torch.argmax(size_logits).item()]
        }

    print(f"Optimal Hyperparameters Discovered: {best_params}")
    return best_params

# Execute the search
# optimal_hyperparameters = optimize_hyperparameters_with_nn(x_train, y_train, K)